# Step 5. Retrievers Benchmark

In [1]:
import os
import re
import pickle
import pandas as pd

from retriever.freq_retriever import FreqRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm.notebook import tqdm

In [2]:
SPLITS_PATH = './splits'
DATA_PATH = './data'

In [18]:
questions = pd.read_csv(os.path.join(DATA_PATH, 'questions.csv'), sep=';')
questions.head()

,question,chunk_id,section_name,chapter_name
0,"Какое звание носит Максим Каммерер, к которому...",037c58a8f1011d66e00e2488bd45380a,СОТРУДНИК КОМКОНА–2 МАКСИМ КАММЕРЕР,1 июня 78–го года
1,"С какой организации, находящейся на Земле, отб...",037c58a8f1011d66e00e2488bd45380a,СОТРУДНИК КОМКОНА–2 МАКСИМ КАММЕРЕР,1 июня 78–го года
2,"Какой предмет извлек Экселенц из ящика стола, ...",037c58a8f1011d66e00e2488bd45380a,СОТРУДНИК КОМКОНА–2 МАКСИМ КАММЕРЕР,1 июня 78–го года
3,Какую инструкцию дал Экселенц Каммереру относи...,037c58a8f1011d66e00e2488bd45380a,СОТРУДНИК КОМКОНА–2 МАКСИМ КАММЕРЕР,1 июня 78–го года
4,"На папке, переданной главному герою, было выти...",9a490f7a02f67db0fa4bbb9710b1e9bb,Я принял папку. С такой я...,1 июня 78–го года


In [19]:
with open(os.path.join(SPLITS_PATH, 'splits.pkl'), 'rb') as f:
    splits = pickle.load(f)
print(len(splits))

99


### Custom FreqRetriever Test

In [20]:
retriever = FreqRetriever.from_documents(
    docs=splits,
    k = 3,
    with_relevance = False
)

In [21]:
def retriever_test(query: str) -> list:
    result = retriever.invoke(query)
    if result:
        return result
    else:
        return "No relevant documents found"

In [22]:
query = """Какое устройство, согласно тексту, Экселенц хотел использовать, чтобы взять Абалкина под контроль, если взять его не удастся?"""
result = retriever_test(query)
if isinstance(result, list):
    print(result[0].metadata)

{'title': 'ЖУК В МУРАВЕЙНИКЕ', 'chapter': '4 июня 78–го года', 'section': 'Но по–видимому даже...', 'id': 'f453eff6231d4c6856fd757243d65d20', 'size': 4466, 'collection': 'beetle_in_anthill'}


In [23]:
def benchmark(retriever, verbose=True):
    num_correct = 0
    hit = 0
    rank = 0
    num_questions = 0
    for i in questions.index:
        
        num_questions += 1
        question = questions.loc[i, 'question']
        correct_chunk_id = questions.loc[i, 'chunk_id']
        result = retriever.invoke(question)

        if result and isinstance(result, list):
            chunks_ids = []
            for doc in result:
                chunks_ids.append(doc.metadata.get('id', ''))
            
            if verbose:
                print(f'Q: {question}')
                print(correct_chunk_id)
                print(chunks_ids)
            if any([idx == correct_chunk_id for idx in chunks_ids]):
                num_correct += 1
                rank += 1./(chunks_ids.index(correct_chunk_id) + 1)
                if chunks_ids.index(correct_chunk_id) == 0:
                    if verbose:
                        print(f'Found First\n')
                
                else:
                    if verbose:
                        print(f'Found\n')
            else:
                if verbose:
                    print(f'Not Found\n')
    
    print(f'Hit Rate: {round(num_correct/num_questions, 3)}')
    print(f'MRR: {round(rank/num_questions, 3)}')
    return None

In [24]:
%%time
benchmark(retriever=retriever, verbose=False)

Hit Rate: 0.862
MRR: 0.747
CPU times: total: 547 ms
Wall time: 549 ms


### Test LangChain BM25Retriever

In [25]:
retriever = BM25Retriever.from_documents(
    splits,
    k = 3,
)

In [26]:
print(retriever)

vectorizer=<rank_bm25.BM25Okapi object at 0x0000020D27E11A90> k=3


In [27]:
%%time
benchmark(retriever=retriever, verbose=False)

Hit Rate: 0.582
MRR: 0.484
CPU times: total: 203 ms
Wall time: 200 ms


### InMemory vector retriever test

In [28]:
embedding = HuggingFaceEmbeddings(model_name="deepvk/USER-bge-m3", model_kwargs={"device": "cpu"})

In [29]:
vector_store = InMemoryVectorStore(embedding)

In [30]:
%%time
for s in tqdm(splits):
    split_id = s.metadata.get('id')
    vector_store.add_documents(documents=[s], ids=[split_id])

  0%|          | 0/99 [00:00<?, ?it/s]

CPU times: total: 19min 20s
Wall time: 3min 20s


In [31]:
retriever = vector_store.as_retriever(k=3)
print(retriever)

tags=['InMemoryVectorStore', 'HuggingFaceEmbeddings'] vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x0000020D27E42F90> search_kwargs={}


In [32]:
%%time
benchmark(retriever=retriever, verbose=False)

Hit Rate: 0.751
MRR: 0.62
CPU times: total: 3min 47s
Wall time: 39.4 s


### Qdrant vector store test

In [ ]:
from qdrant_client.http.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

In [ ]:
QDRANT_HOST = "127.0.0.1"
QDRANT_PORT = 6333
QDRANT_EMBEDDING_SIZE = 1024
QDRANT_DISTANCE = Distance.COSINE

In [ ]:
qdrant_client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

In [ ]:
qdrant_client.create_collection(
    collection_name='virtual_light',
    vectors_config=VectorParams(size=QDRANT_EMBEDDING_SIZE, distance=QDRANT_DISTANCE),
)

In [ ]:
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name='virtual_light',
    distance=QDRANT_DISTANCE,
    embedding=embedding,
)

In [ ]:
%%time
for s in tqdm(splits):
    split_id = s.metadata.get('id')
    vector_store.add_documents(documents=[s], ids=[split_id])    

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(retriever)

In [ ]:
result = retriever_test(query)
if isinstance(result, list):
    print(result[0].metadata)
    print(result[1].metadata)
    print(result[2].metadata)

In [ ]:
%%time
benchmark(retriever=retriever, verbose=False)

In [ ]:
!pip list

In [ ]:
!pip show notebook